# IEEE-CIS Fraud Detection - Team 1 Member 1 Colab Setup

This notebook prepares a Colab-friendly preprocessing pipeline for **Team 1 / Member 1**:

- Download IEEE-CIS data with the Kaggle API
- Load each raw CSV exactly once
- Downcast memory usage immediately after load
- Engineer `TransactionDT` time-based features
- Clean `Identity` categorical columns for downstream modeling
- Merge transaction and identity tables
- Export reusable `.parquet` outputs to Google Drive


## 1. Runtime setup

Before running the notebook:

1. In Colab, set runtime to **GPU**.
2. Prepare your Kaggle API token file: `kaggle.json`.
3. Run cells from top to bottom.


In [ ]:
from pathlib import Path

RANDOM_SEED = 42
TARGET_COLUMN = "isFraud"
KAGGLE_DATASET = "ieee-fraud-detection"

# Edit only this section when adapting the notebook.
PROJECT_ROOT = Path("/content/drive/MyDrive/ieee_cis_fraud/team1_member1")
RAW_DATA_DIR = PROJECT_ROOT / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "interim"

OUTPUT_TRAIN_FILE = INTERIM_DATA_DIR / "train_member1_features.parquet"
OUTPUT_TEST_FILE = INTERIM_DATA_DIR / "test_member1_features.parquet"
FEATURE_MANIFEST_FILE = INTERIM_DATA_DIR / "feature_manifest.json"

PROJECT_ROOT, RAW_DATA_DIR, INTERIM_DATA_DIR

(PosixPath('/content/drive/MyDrive/ieee_cis_fraud/team1_member1'),
 PosixPath('/content/drive/MyDrive/ieee_cis_fraud/team1_member1/raw'),
 PosixPath('/content/drive/MyDrive/ieee_cis_fraud/team1_member1/interim'))

In [ ]:
!pip -q install kaggle polars pyarrow fastparquet scikit-learn xgboost lightgbm catboost optuna shap seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 29.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_DATA_DIR: {RAW_DATA_DIR}")
print(f"INTERIM_DATA_DIR: {INTERIM_DATA_DIR}")

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/ieee_cis_fraud/team1_member1
RAW_DATA_DIR: /content/drive/MyDrive/ieee_cis_fraud/team1_member1/raw
INTERIM_DATA_DIR: /content/drive/MyDrive/ieee_cis_fraud/team1_member1/interim


## 2. Kaggle API bootstrap

Run the next cell, upload your `kaggle.json`, and the notebook will place it in the expected Kaggle location.

In [ ]:
import os
from google.colab import files

uploaded = files.upload()
assert 'kaggle.json' in uploaded, 'Please upload kaggle.json to continue.'

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])

!chmod 600 /root/.kaggle/kaggle.json
print('Kaggle credentials configured.')

Saving kaggle.json to kaggle.json
Kaggle credentials configured.


In [ ]:
import zipfile
from pathlib import Path

dataset_slug = KAGGLE_DATASET
download_dir = Path('/content/ieee_download')
download_dir.mkdir(parents=True, exist_ok=True)

!kaggle competitions download -c {dataset_slug} -p /content/ieee_download

zip_path = download_dir / f"{dataset_slug}.zip"
assert zip_path.exists(), f"Expected archive not found: {zip_path}"

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(RAW_DATA_DIR)

expected_files = [
    'train_transaction.csv',
    'train_identity.csv',
    'test_transaction.csv',
    'test_identity.csv',
]

missing = [name for name in expected_files if not (RAW_DATA_DIR / name).exists()]
assert not missing, f"Missing expected files after unzip: {missing}"

print('Download and extraction complete.')
for name in expected_files:
    print('-', RAW_DATA_DIR / name)

100% 118M/118M [00:00<00:00, 140MB/s]

Download and extraction complete.
- /content/drive/MyDrive/ieee_cis_fraud/team1_member1/raw/train_transaction.csv
- /content/drive/MyDrive/ieee_cis_fraud/team1_member1/raw/train_identity.csv
- /content/drive/MyDrive/ieee_cis_fraud/team1_member1/raw/test_transaction.csv
- /content/drive/MyDrive/ieee_cis_fraud/team1_member1/raw/test_identity.csv


## 3. Feature engineering strategy

This notebook intentionally focuses on **Member 1 scope**:

- `TransactionDT` time features only
- Identity-table cleanup
- Stable categorical preparation without one-hot encoding
- Parquet outputs for downstream teams


In [ ]:
import gc
import json
import math
import re
from typing import Iterable

import numpy as np
import pandas as pd
import polars as pl


def memory_usage_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / 1024 ** 2


def downcast_numeric_frame(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()

    int_cols = result.select_dtypes(include=['int', 'int64', 'int32', 'int16']).columns
    float_cols = result.select_dtypes(include=['float', 'float64', 'float32']).columns

    for col in int_cols:
        result[col] = pd.to_numeric(result[col], downcast='integer')

    for col in float_cols:
        result[col] = pd.to_numeric(result[col], downcast='float')

    return result


def normalize_categorical_text(series: pd.Series) -> pd.Series:
    series = series.astype('string')
    series = series.fillna('__MISSING__')
    series = series.str.strip()
    series = series.str.lower()
    series = series.str.replace(r'\s+', '_', regex=True)
    series = series.str.replace(r'[^0-9a-zA-Z_\-]', '_', regex=True)
    return series.astype('string')


def add_transactiondt_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    seconds = result['TransactionDT'].astype('float32')

    result['TransactionDT_days'] = (seconds / 86400).astype('float32')
    result['TransactionDT_hours'] = (seconds / 3600).astype('float32')
    result['TransactionDT_day_index'] = np.floor(seconds / 86400).astype('int32')
    result['TransactionDT_hour_of_day'] = np.floor((seconds % 86400) / 3600).astype('int8')
    result['TransactionDT_day_of_week_proxy'] = (result['TransactionDT_day_index'] % 7).astype('int8')
    result['TransactionDT_is_weekend_proxy'] = result['TransactionDT_day_of_week_proxy'].isin([5, 6]).astype('int8')
    result['TransactionDT_week_index'] = np.floor(seconds / (86400 * 7)).astype('int16')

    return result


def clean_identity_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    categorical_cols = result.select_dtypes(include=['object', 'string']).columns.tolist()

    for col in categorical_cols:
        result[col] = normalize_categorical_text(result[col])

    numeric_cols = [col for col in result.columns if col != 'TransactionID' and col not in categorical_cols]
    for col in numeric_cols:
        if pd.api.types.is_numeric_dtype(result[col]):
            result[col] = result[col].fillna(-1)

    result = downcast_numeric_frame(result)
    return result


def read_csv_once(path: Path, name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    before = memory_usage_mb(df)
    df = downcast_numeric_frame(df)
    after = memory_usage_mb(df)
    print(f"{name}: {before:.2f} MB -> {after:.2f} MB")
    return df


def dataframe_summary(name: str, df: pd.DataFrame) -> None:
    print(f"{name}: shape={df.shape}, memory={memory_usage_mb(df):.2f} MB")
    print(df.head(2))


In [ ]:
train_transaction = read_csv_once(RAW_DATA_DIR / 'train_transaction.csv', 'train_transaction')
train_identity = read_csv_once(RAW_DATA_DIR / 'train_identity.csv', 'train_identity')
test_transaction = read_csv_once(RAW_DATA_DIR / 'test_transaction.csv', 'test_transaction')
test_identity = read_csv_once(RAW_DATA_DIR / 'test_identity.csv', 'test_identity')

dataframe_summary('train_transaction', train_transaction)
dataframe_summary('train_identity', train_identity)
dataframe_summary('test_transaction', test_transaction)
dataframe_summary('test_identity', test_identity)

train_transaction: 2062.07 MB -> 1203.22 MB
train_identity: 143.14 MB -> 129.94 MB
test_transaction: 1771.84 MB -> 1038.31 MB
test_identity: 140.08 MB -> 127.09 MB
train_transaction: shape=(590540, 394), memory=1203.22 MB
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   

   card2  card3       card4  card5  ... V330  V331  V332  V333  V334 V335  \
0    NaN  150.0    discover  142.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
1  404.0  150.0  mastercard  102.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   

  V336  V337  V338  V339  
0  NaN   NaN   NaN   NaN  
1  NaN   NaN   NaN   NaN  

[2 rows x 394 columns]
train_identity: shape=(144233, 41), memory=129.94 MB
   TransactionID  id_01    id_02  id_03  id_04  id_05  id_06  id_07  id_08  \
0        2987004    0.0  70787.0    NaN    NaN    NaN    NaN    NaN    NaN   


In [ ]:
train_transaction = add_transactiondt_features(train_transaction)
test_transaction = add_transactiondt_features(test_transaction)

transaction_time_features = [
    'TransactionDT',
    'TransactionDT_days',
    'TransactionDT_hours',
    'TransactionDT_day_index',
    'TransactionDT_hour_of_day',
    'TransactionDT_day_of_week_proxy',
    'TransactionDT_is_weekend_proxy',
    'TransactionDT_week_index',
]

print('Added TransactionDT features:')
print(transaction_time_features)

Added TransactionDT features:
['TransactionDT', 'TransactionDT_days', 'TransactionDT_hours', 'TransactionDT_day_index', 'TransactionDT_hour_of_day', 'TransactionDT_day_of_week_proxy', 'TransactionDT_is_weekend_proxy', 'TransactionDT_week_index']


In [ ]:
train_identity = clean_identity_features(train_identity)
test_identity = clean_identity_features(test_identity)

# Align test identity column names with train before merging.
test_identity.columns = [c.replace('-', '_') if 'id-' in c else c for c in test_identity.columns]

identity_categorical_cols = train_identity.select_dtypes(include=['object', 'string']).columns.tolist()
print(f'Identity categorical columns cleaned: {len(identity_categorical_cols)}')
print(identity_categorical_cols[:20])

Identity categorical columns cleaned: 17
['id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [ ]:
train_transaction_pl = pl.from_pandas(train_transaction[['TransactionID'] + transaction_time_features + [TARGET_COLUMN]])
print(train_transaction_pl.select([
    pl.len().alias('rows'),
    pl.col('TransactionDT_hour_of_day').n_unique().alias('unique_hours'),
    pl.col('TransactionDT_day_of_week_proxy').n_unique().alias('unique_day_proxy')
]))

shape: (1, 3)
┌────────┬──────────────┬──────────────────┐
│ rows   ┆ unique_hours ┆ unique_day_proxy │
│ ---    ┆ ---          ┆ ---              │
│ u32    ┆ u32          ┆ u32              │
╞════════╪══════════════╪══════════════════╡
│ 590540 ┆ 24           ┆ 7                │
└────────┴──────────────┴──────────────────┘


In [ ]:
train_merged = train_transaction.merge(train_identity, on='TransactionID', how='left', validate='one_to_one')
test_merged = test_transaction.merge(test_identity, on='TransactionID', how='left', validate='one_to_one')

assert train_merged['TransactionID'].is_unique, 'Duplicate TransactionID found in train merge.'
assert test_merged['TransactionID'].is_unique, 'Duplicate TransactionID found in test merge.'

print('Merge completed.')
print('train_merged shape:', train_merged.shape)
print('test_merged shape:', test_merged.shape)

Merge completed.
train_merged shape: (590540, 441)
test_merged shape: (506691, 440)


In [ ]:
assert len(train_merged) == len(train_transaction), 'Train row count changed after merge.'
assert len(test_merged) == len(test_transaction), 'Test row count changed after merge.'
assert TARGET_COLUMN in train_merged.columns, 'Target column missing from train output.'
assert TARGET_COLUMN not in test_merged.columns, 'Target column should not exist in test output.'

shared_columns = sorted(set(train_merged.columns) - {TARGET_COLUMN})
test_columns = sorted(test_merged.columns)
assert shared_columns == test_columns, f'Inconsistent columns. Unique to train: {set(shared_columns) - set(test_columns)}. Unique to test: {set(test_columns) - set(shared_columns)}'

del train_transaction, train_identity, test_transaction, test_identity
gc.collect()

print('Row count and schema checks passed.')

Row count and schema checks passed.


In [ ]:
train_merged.to_parquet(OUTPUT_TRAIN_FILE, index=False)
test_merged.to_parquet(OUTPUT_TEST_FILE, index=False)

feature_manifest = {
    'target_column': TARGET_COLUMN,
    'join_key': 'TransactionID',
    'train_output': str(OUTPUT_TRAIN_FILE),
    'test_output': str(OUTPUT_TEST_FILE),
    'transactiondt_features': transaction_time_features,
    'identity_categorical_columns': identity_categorical_cols,
    'train_columns': train_merged.columns.tolist(),
    'test_columns': test_merged.columns.tolist(),
    'train_shape': list(train_merged.shape),
    'test_shape': list(test_merged.shape),
}

with open(FEATURE_MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(feature_manifest, f, indent=2)

print('Exports written:')
print('-', OUTPUT_TRAIN_FILE)
print('-', OUTPUT_TEST_FILE)
print('-', FEATURE_MANIFEST_FILE)

Exports written:
- /content/drive/MyDrive/ieee_cis_fraud/team1_member1/interim/train_member1_features.parquet
- /content/drive/MyDrive/ieee_cis_fraud/team1_member1/interim/test_member1_features.parquet
- /content/drive/MyDrive/ieee_cis_fraud/team1_member1/interim/feature_manifest.json


In [ ]:
train_reloaded = pd.read_parquet(OUTPUT_TRAIN_FILE)
test_reloaded = pd.read_parquet(OUTPUT_TEST_FILE)

assert train_reloaded.shape == train_merged.shape
assert test_reloaded.shape == test_merged.shape

print('Parquet reload validation passed.')
print(train_reloaded[transaction_time_features + ['TransactionID', TARGET_COLUMN]].head())

Parquet reload validation passed.
   TransactionDT  TransactionDT_days  TransactionDT_hours  \
0          86400            1.000000            24.000000   
1          86401            1.000012            24.000278   
2          86469            1.000799            24.019167   
3          86499            1.001146            24.027500   
4          86506            1.001227            24.029444   

   TransactionDT_day_index  TransactionDT_hour_of_day  \
0                        1                          0   
1                        1                          0   
2                        1                          0   
3                        1                          0   
4                        1                          0   

   TransactionDT_day_of_week_proxy  TransactionDT_is_weekend_proxy  \
0                                1                               0   
1                                1                               0   
2                                1            

## 4. Handoff notes

The generated parquet files are intended as reusable inputs for later teams.

- `train_member1_features.parquet` keeps `isFraud`
- `test_member1_features.parquet` excludes `isFraud`
- `feature_manifest.json` records the final feature set and output paths

This notebook does **not** perform one-hot encoding or model-specific target encoding, so downstream ML and DL members can choose their own encoding strategy.
